# HW6: Interest-Rate Products and Valuation

The FRA product, factory builder, and valuation engine are complete and provided
as a worked example. Study this example, then follow the same structure to
implement a vanilla RFR swap and an overnight index basis swap.

1. Read `ProductFRA` in `fixedincomelib/product/linear_products.py`,
   `ProductFactory.create_fra` in `fixedincomelib/product/product_factory.py`,
   and `ValuationEngineProductFra` in `fixedincomelib/yield_curve/valuation_engine.py`.
2. Complete the six `#TODO` blocks listed below. Construct each product's legs,
   create products from their conventions, and combine the leg values and cash.
3. Reuse the supplied `ProductInterestRateStream` and `ValuationEngineInterestRateStream`.
   The factory dispatcher, tokenizer, registrations, and valuation-engine constructors are also supplied.
4. Assume all inputs are valid. Use a Python 3.11 kernel with `requirements.txt` installed.
   After editing the library, **restart the kernel and run all cells** from top to bottom.
   Check the displayed cashflows and PV comparisons.

| # | File | Class / method | `#TODO` line | Task |
|---|---|---|---:|---|
| 1 | `fixedincomelib/product/linear_products.py` | `ProductRFRSwap.__init__` | 532 | Construct the fixed and overnight legs. |
| 2 | `fixedincomelib/product/linear_products.py` | `ProductOvernightIndexBasisSwap.__init__` | 730 | Construct both overnight legs; add spread to leg 1. |
| 3 | `fixedincomelib/product/product_factory.py` | `ProductFactory.create_rfr_swap` | 44 | Create the swap from its convention, dates, and quote. |
| 4 | `fixedincomelib/product/product_factory.py` | `ProductFactory.create_overnight_index_basis_swap` | 57 | Create the basis swap from both index conventions and spread. |
| 5 | `fixedincomelib/yield_curve/valuation_engine.py` | `ValuationEngineProductRfrSwap.calculate_value` | 163 | Combine leg PVs and cash using the supplied signs. |
| 6 | `fixedincomelib/yield_curve/valuation_engine.py` | `ValuationEngineProductOvernightIndexBasisSwap.calculate_value` | 218 | Combine the three signed component PVs and cash. |

Line numbers refer to the original files; they will move as you add code.
Search for `#TODO` or the method name after editing.

The FRA fixes a single term rate at the accrual start and pays at the accrual end.
Swap floating legs accumulate overnight rates; use the appropriate stream type
when following the FRA example. Project each floating leg with its index curve
and discount payments with the funding curve.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from fixedincomelib import (
    qfCreateData1D, qfCreateDataCollection,
    qfCreateBuildMethod, qfCreateModelBuildMethodCollection, qfCreateModel,
    qfCreateProductInterestRateStream, qfCreateProductFRA,
    qfCreateProductRFRSwap, qfCreateProductOvernightIndexBasisSwap,
    qfCreateProductFromDataConvention, qfDisplayProduct,
    qfCreateValuationParameters, qfCreateValuationParametersCollection,
    qfCreateValueReport, qfCreateCashflowsReport, qfInsertTermRateFixing,
)
from fixedincomelib.date import Date
from fixedincomelib.market.registries import DataConventionRegistry
from fixedincomelib.valuation import ValuationRequest
from fixedincomelib.yield_curve.valuation_engine import (
    ValuationEngineInterestRateStream, ValuationEngineProductFra,
    ValuationEngineProductRfrSwap, ValuationEngineProductOvernightIndexBasisSwap,
)

np.set_printoptions(precision=6, suppress=True)
pd.options.display.float_format = "{:,.6f}".format
pd.options.display.max_colwidth = None

## 1. Input data

Rates are annual decimals: `0.04 = 4%`, `0.001 = 10 bp`. These are synthetic teaching inputs.
SOFR and Fed Funds have separate projection curves. The funding component adds an IFR spread
on top of SOFR, as in HW5. Edit the inputs here and rerun the following cells.

In [2]:
VALUE_DATE = "2026-01-02"
EFFECTIVE_DATE = "2026-04-02"
FRA_END_DATE = "2026-07-02"
SWAP_END_DATE = "2027-04-02"
FUNDING_INDEX = "SOFR-1B-FLAT"
NOTIONAL = 1_000_000
FIXED_RATE = 0.04
BASIS_SPREAD = 0.001
BASIS_CONVENTION = "USD-FF-3M-OVER-USD-SOFR-OIS-3M"

sofr_quotes = pd.DataFrame(
    {"values": [0.03, 0.04, 0.05]},
    index=pd.Index(["2026-04-02", "2027-01-04", "2028-01-03"], name="Node date"),
)
ff_quotes = pd.DataFrame(
    {"values": [0.028, 0.037]},
    index=pd.Index(["2026-07-02", "2028-01-03"], name="Node date"),
)
funding_quotes = pd.DataFrame(
    {"values": [0.0, 0.001]},
    index=pd.Index(["2026-07-02", "2028-01-03"], name="Node date"),
)

display(sofr_quotes.rename(columns={"values": "SOFR-1B IFR"}))
display(ff_quotes.rename(columns={"values": "FF-1B IFR"}))
display(funding_quotes.rename(columns={"values": "Funding IFR spread"}))

,SOFR-1B IFR
Node date,
2026-04-02,0.030000
2027-01-04,0.040000
2028-01-03,0.050000


,FF-1B IFR
Node date,
2026-07-02,0.028000
2028-01-03,0.037000


,Funding IFR spread
Node date,
2026-07-02,0.000000
2028-01-03,0.001000


## 2. Data and build methods

Create the IFR datasets and build methods, with SOFR first, Fed Funds second and funding third.
The model construction copies these IFR inputs into state variables, as in HW5.

In [3]:
sofr_data = qfCreateData1D("INSTANTANEOUS FORWARD RATE", "SOFR-1B-IFR", sofr_quotes)
ff_data = qfCreateData1D("INSTANTANEOUS FORWARD RATE", "FF-1B-IFR", ff_quotes)
funding_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE", "SOFR-1B-FUNDING-IFR", funding_quotes,
)
data_collection = qfCreateDataCollection([sofr_data, ff_data, funding_data])
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-IFR
1,DATA1D,INSTANTANEOUS FORWARD RATE,FF-1B-IFR
2,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-FUNDING-IFR


In [4]:
sofr_method = qfCreateBuildMethod("YIELD_CURVE_INDEX", {
    "TARGET": "SOFR-1B", "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR",
})
ff_method = qfCreateBuildMethod("YIELD_CURVE_INDEX", {
    "TARGET": "FF-1B", "INSTANTANEOUS FORWARD RATE": "FF-1B-IFR",
})
funding_method = qfCreateBuildMethod("YIELD_CURVE_FUNDING", {
    "TARGET": FUNDING_INDEX, "REFERENCE INDEX": "SOFR-1B",
    "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR",
})
build_methods = qfCreateModelBuildMethodCollection([sofr_method, ff_method, funding_method])

## 3. Build the model

The projection index selects the curve used to estimate a floating coupon.
`FundingIndexParameter` selects the curve used to discount both legs on their payment dates.

In [5]:
model = qfCreateModel(VALUE_DATE, "YIELD_CURVE", data_collection, build_methods)
funding_parameter = qfCreateValuationParameters(
    "FUNDING INDEX PARAMETER", {"FUNDING INDEX": FUNDING_INDEX},
)
parameters = qfCreateValuationParametersCollection([funding_parameter])

print("Component sizes:", model.num_sub_components)
print("Component order:", model.component_indices)
print("SOFR DF:", model.discount_factor("SOFR-1B", Date(FRA_END_DATE)))
print("Funding DF:", model.discount_factor(FUNDING_INDEX, Date(FRA_END_DATE)))

Component sizes: [3, 2, 2]
Component order: {'SOFRON Actual/360': 0, 'FedFundsON Actual/360': 1, 'SOFR-1B-FLAT': 2}
SOFR DF: 0.9827801233868858
Funding DF: 0.9827801233868858


## 4. Interest-rate streams

`ProductInterestRateStream` is the supplied leg helper. A fixed coupon pays $N K \alpha$;
an overnight coupon pays $N\alpha(R+s)$, with spread added outside daily compounding.
The existing fixed-accrued cashflow still represents $N\alpha$; the stream applies $K$ once.

`ProductInterestRateStream` and `ValuationEngineInterestRateStream` are supplied in full.
Use `accrual_period=None` for a single coupon and `is_on_index=False` for a term-rate coupon.


In [6]:
fixed_stream = qfCreateProductInterestRateStream(
    EFFECTIVE_DATE, SWAP_END_DATE, "1Y", NOTIONAL, "USD", "ACT/360",
    fixed_rate=FIXED_RATE, payment_offset="2D",
)
overnight_stream = qfCreateProductInterestRateStream(
    EFFECTIVE_DATE, SWAP_END_DATE, "1Y", NOTIONAL, "USD", "ACT/360",
    holiday_convention="SOFR", float_index="SOFR-1B", payment_offset="2D",
)

display(qfDisplayProduct(fixed_stream))
display(qfCreateCashflowsReport(model, fixed_stream, parameters).display())
display(qfCreateCashflowsReport(model, overnight_stream, parameters).display())

,Name,Value
0,Product Type,PRODUCT_INTEREST_RATE_STREAM
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-04-02
5,Termination Date,2027-04-02
6,Last Payment Date,2027-04-06
7,Number of Cashflows,1
8,Fixed Rate,0.040000
9,Floating Index,None


,Leg,Currency,Accrual Start,Accrual End,Payment Date,Notional,Accrual,Rate,Spread,Cashflow,Discount Factor,PV
0,Stream,USD,2026-04-02,2027-04-02,2027-04-06,1000000,1.013889,0.040000,0.000000,"40,555.555556",0.950167,"38,534.539463"


,Leg,Currency,Accrual Start,Accrual End,Payment Date,Notional,Accrual,Rate,Spread,Cashflow,Discount Factor,PV
0,Stream,USD,2026-04-02,2027-04-02,2027-04-06,1000000,1.013889,0.042730,0.000000,"43,323.153601",0.950167,"41,164.219038"


In [7]:
stream_engine = ValuationEngineInterestRateStream(
    model, parameters, overnight_stream, ValuationRequest.PV,
)
stream_engine.calculate_value()
print("Floating stream PV:", stream_engine.value)
print("Cash today:", stream_engine.cash)

Floating stream PV: 41164.21903793787
Cash today: 0.0


## 5. FRA

The teaching FRA fixes the whole-period simple rate at $T_s$ and settles at $T_e$.
Before fixing, use the SOFR projection curve to calculate
$F=(P_p(t,T_s)/P_p(t,T_e)-1)/\tau$.
For receive-fixed, $PV=N\tau(K-F)D_f(t,T_e)$. `pay` reverses the sign.
The projection curve identifies a teaching term rate; its fixing is stored separately from daily SOFR fixings.

The FRA is supplied in full. Read `ProductFRA.__init__`, `ProductFactory.create_fra`,
and `ValuationEngineProductFra.calculate_value` to see how product construction,
convention-driven creation, and leg valuation fit together. Use these implementations
as the starting point for the two swap products.


In [8]:
fra = qfCreateProductFRA(
    EFFECTIVE_DATE, FRA_END_DATE, "SOFR-1B", FIXED_RATE, "receive", NOTIONAL,
    accrual_basis="ACT/360",
)
display(qfDisplayProduct(fra))
display(qfCreateCashflowsReport(model, fra, parameters).display())

,Name,Value
0,Product Type,PRODUCT_FRA
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-04-02
5,Termination Date,2026-07-02
6,Fixing Date,2026-04-02
7,Payment Date,2026-07-02
8,Projection Index,SOFR-1B
9,Fixed Rate,0.040000


,Leg,Currency,Accrual Start,Accrual End,Payment Date,Notional,Accrual,Rate,Spread,Cashflow,Discount Factor,PV
0,Fixed,USD,2026-04-02,2026-07-02,2026-07-02,"1,000,000.000000",0.252778,0.040000,0.000000,"10,111.111111",0.982780,"9,936.999025"
1,Floating,USD,2026-04-02,2026-07-02,2026-07-02,"-1,000,000.000000",0.252778,0.039649,0.000000,"-10,022.494856",0.982780,"-9,849.908731"


In [9]:
df_start = model.discount_factor("SOFR-1B", fra.effective_date)
df_end = model.discount_factor("SOFR-1B", fra.termination_date)
forward_rate = (df_start / df_end - 1.0) / fra.accrued
expected_fra_pv = (
    NOTIONAL * fra.accrued * (FIXED_RATE - forward_rate)
    * model.discount_factor(FUNDING_INDEX, fra.payment_date)
)
fra_engine = ValuationEngineProductFra(model, parameters, fra, ValuationRequest.PV)
fra_engine.calculate_value()
print("Projected term rate:", forward_rate)
print("FRA PV / formula:", fra_engine.value, expected_fra_pv)
np.testing.assert_allclose(fra_engine.value, expected_fra_pv, atol=1e-8)

Projected term rate: 0.03964943019925975
FRA PV / formula: 87.09029420688239 87.09029420688128


Once $T_s$ is reached, supply the observed term fixing and discount the known payment.
The following fixing is synthetic.


In [10]:
fixing_model = qfCreateModel(EFFECTIVE_DATE, "YIELD_CURVE", data_collection, build_methods)
observed_term_rate = 0.045
qfInsertTermRateFixing("SOFR-1B", EFFECTIVE_DATE, FRA_END_DATE, "ACT/360", observed_term_rate)
fixed_fra_pv = qfCreateValueReport(fixing_model, fra, parameters, "PV")[0][1]
known_amount = NOTIONAL * fra.accrued * (FIXED_RATE - observed_term_rate)
print("Known amount at Te:", known_amount)
print("PV on Ts:", fixed_fra_pv)
np.testing.assert_allclose(
    fixed_fra_pv, known_amount * fixing_model.discount_factor(FUNDING_INDEX, fra.payment_date),
    atol=1e-8,
)

Known amount at Te: -1263.8888888888882
PV on Ts: -1251.3472673390097


## 6. Vanilla RFR swap

The template's vanilla swap is fixed versus compounded overnight interest.
`receive` means receive the fixed leg and pay the floating leg.
Each leg uses the stream helper; the swap engine combines their PV and cash.
Unlike the FRA term coupon, an overnight coupon accumulates fixings throughout its accrual period.

In `fixedincomelib/product/linear_products.py`, complete the `#TODO` in
`ProductRFRSwap.__init__`: set `fixed_leg_` and `floating_leg_` using
`InterestRateStream`. Use each leg's accrual period and the supplied payment conventions.
The floating leg uses the overnight index, compounding method, and spread.

In `fixedincomelib/yield_curve/valuation_engine.py`, complete `calculate_value`
in `ValuationEngineProductRfrSwap`. Use the supplied leg engines and signs to
set `self.value_` and `self.cash_`.


In [11]:
swap = qfCreateProductRFRSwap(
    effective_date=EFFECTIVE_DATE, term_or_termination_date=SWAP_END_DATE,
    payment_off_set="2D", on_index="SOFR-1B", fixed_rate=FIXED_RATE,
    pay_or_rec="receive", notional=NOTIONAL,
    accrual_period="1Y", accrual_basis="ACT/360",
    floating_leg_accrual_period="1Y",
)
display(qfDisplayProduct(swap))
display(qfCreateCashflowsReport(model, swap, parameters).display())

,Name,Value
0,Product Type,PRODUCT_RFR_SWAP
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-04-02
5,Termination Date,2027-04-02
6,Last Payment Date,2027-04-06
7,ON Index,SOFR-1B
8,Fixed Rate,0.040000
9,Spread,0.000000


,Leg,Currency,Accrual Start,Accrual End,Payment Date,Notional,Accrual,Rate,Spread,Cashflow,Discount Factor,PV
0,Fixed,USD,2026-04-02,2027-04-02,2027-04-06,"1,000,000.000000",1.013889,0.040000,0.000000,"40,555.555556",0.950167,"38,534.539463"
1,Floating,USD,2026-04-02,2027-04-02,2027-04-06,"-1,000,000.000000",1.013889,0.042730,0.000000,"-43,323.153601",0.950167,"-41,164.219038"


In [12]:
swap_engine = ValuationEngineProductRfrSwap(model, parameters, swap, ValuationRequest.PV)
swap_engine.calculate_value()
fixed_pv = qfCreateValueReport(model, swap.fixed_leg, parameters, "PV")[0][1]
floating_pv = qfCreateValueReport(model, swap.floating_leg, parameters, "PV")[0][1]
print("Fixed / floating PV:", fixed_pv, floating_pv)
print("Swap PV:", swap_engine.value)
np.testing.assert_allclose(swap_engine.value, fixed_pv - floating_pv, atol=1e-8)

Fixed / floating PV: 38534.53946314968 41164.21903793787
Swap PV: -2629.6795747881915


## 7. Overnight index basis swap

The template convention specifies leg 1 as SOFR plus spread and leg 2 as Fed Funds.
`receive` means receive leg 1 and pay leg 2. Each index uses its own projection curve;
both legs use the same USD funding curve. The convention's original template name is retained.

In `fixedincomelib/product/linear_products.py`, complete the `#TODO` in
`ProductOvernightIndexBasisSwap.__init__`: set `floating_leg_1_` and
`floating_leg_2_`. Use each leg's index and accrual period; apply `spread_` only to leg 1.

In `fixedincomelib/yield_curve/valuation_engine.py`, complete `calculate_value`
in `ValuationEngineProductOvernightIndexBasisSwap`. The supplied `_legs()` gives
the leg-1 index, leg-1 spread, and leg-2 index engines with their signs.
Set `self.value_` and `self.cash_` to their signed sums.


In [13]:
basis_swap = qfCreateProductOvernightIndexBasisSwap(
    effective_date=EFFECTIVE_DATE, term_or_termination_date=SWAP_END_DATE,
    payment_off_set="2D", on_index_leg_1="SOFR-1B", on_index_leg_2="FF-1B",
    spread_over_leg_1=BASIS_SPREAD, pay_or_rec_leg_1="receive", notional=NOTIONAL,
    accrual_period_1="3M", accrual_period_2="3M", accrual_basis="ACT/360",
)
display(qfDisplayProduct(basis_swap))
display(qfCreateCashflowsReport(model, basis_swap, parameters).display())

,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-04-02
5,Termination Date,2027-04-02
6,Last Payment Date,2027-04-06
7,ON Index Leg 1,SOFR-1B
8,ON Index Leg 2,FF-1B
9,Spread Over Leg 1,0.001000


,Leg,Currency,Accrual Start,Accrual End,Payment Date,Notional,Accrual,Rate,Spread,Cashflow,Discount Factor,PV
0,Leg 1 index,USD,2026-04-02,2026-07-02,2026-07-07,"1,000,000.000000",0.252778,0.039649,0.000000,"10,022.494856",0.982228,"9,844.378144"
1,Leg 1 index,USD,2026-07-02,2026-10-02,2026-10-06,"1,000,000.000000",0.255556,0.039652,0.000000,"10,133.188318",0.972239,"9,851.882801"
2,Leg 1 index,USD,2026-10-02,2027-01-04,2027-01-06,"1,000,000.000000",0.261111,0.039656,0.000000,"10,354.611638",0.962191,"9,963.112508"
3,Leg 1 index,USD,2027-01-04,2027-04-02,2027-04-06,"1,000,000.000000",0.244444,0.049614,0.000000,"12,127.746401",0.950167,"11,523.381098"
4,Leg 1 spread,USD,2026-04-02,2026-07-02,2026-07-07,"1,000,000.000000",0.252778,0.001000,0.000000,252.777778,0.982228,248.285488
5,Leg 1 spread,USD,2026-07-02,2026-10-02,2026-10-06,"1,000,000.000000",0.255556,0.001000,0.000000,255.555556,0.972239,248.461126
6,Leg 1 spread,USD,2026-10-02,2027-01-04,2027-01-06,"1,000,000.000000",0.261111,0.001000,0.000000,261.111111,0.962191,251.238720
7,Leg 1 spread,USD,2027-01-04,2027-04-02,2027-04-06,"1,000,000.000000",0.244444,0.001000,0.000000,244.444444,0.950167,232.262978
8,Leg 2 index,USD,2026-04-02,2026-07-02,2026-07-07,"-1,000,000.000000",0.252778,0.027713,0.000000,"-7,005.244652",0.982228,"-6,880.749588"
9,Leg 2 index,USD,2026-07-02,2026-10-02,2026-10-06,"-1,000,000.000000",0.255556,0.036664,0.000000,"-9,369.650295",0.972239,"-9,109.541212"


In [14]:
basis_engine = ValuationEngineProductOvernightIndexBasisSwap(
    model, parameters, basis_swap, ValuationRequest.PV,
)
basis_engine.calculate_value()
leg_1_pv = qfCreateValueReport(model, basis_swap.floating_leg_1, parameters, "PV")[0][1]
leg_2_pv = qfCreateValueReport(model, basis_swap.floating_leg_2, parameters, "PV")[0][1]
print("Leg 1 / leg 2 PV:", leg_1_pv, leg_2_pv)
print("Basis swap PV:", basis_engine.value)
np.testing.assert_allclose(basis_engine.value, leg_1_pv - leg_2_pv, atol=1e-8)

Leg 1 / leg 2 PV: 42163.002863421294 33716.53039225433
Basis swap PV: 8446.472471166962


## 8. Product factory

`qfCreateProductFromDataConvention` looks up the convention and dispatches to its registered builder.
The quote is the fixed rate for FRA/swap and the spread over leg 1 for the basis swap.

| Product | `axis1` | Meaning |
|---|---|---|
| FRA | `3Mx6M` | Start in 3 months, end in 6 months, both from `value_date` |
| FRA | `2026-04-02x2026-07-02` | Explicit start and end |
| Swap / basis swap | `3Mx1Y` | Start in 3 months, then run for 1 year |
| Swap / basis swap | `1Y` | Spot-starting, run for 1 year |

`ProductFactory.create_fra` is the worked builder example. The common dispatcher
`create_product_from_data_convention`, tokenizer, and registrations are supplied.
Complete the two `#TODO` blocks in `fixedincomelib/product/product_factory.py`:

- `create_rfr_swap`: read the swap convention and create fixed and overnight legs
  through `ProductRFRSwap`; the quote is the fixed rate.
- `create_overnight_index_basis_swap`: read both indices and their accrual periods,
  then create `ProductOvernightIndexBasisSwap`; the quote is the spread on leg 1.

Follow the FRA builder's parameter mapping, but use the swap date rules in the table above.


In [15]:
display(DataConventionRegistry().get("USD-SOFR-FRA").display())
display(DataConventionRegistry().get("USD-SOFR-OIS").display())

,Name,Value
0,INDEX,SOFR-1B
1,ACCRUAL_BASIS,ACT/360
2,BUSINESS_DAY_CONVENTION,F
3,HOLIDAY_CONVENTION,USGS
4,FIXING_OFFSET,0D
5,PAYMENT_OFFSET,0D


,Name,Value
0,INDEX,SOFR-1B
1,ACCRUAL_BASIS,ACT/360
2,ACCRUAL_PERIOD,1Y
3,PAYMENT_OFFSET,2D
4,PAYMENT_BUSINESS_DAY_CONVENTION,F
5,PAYMENT_HOLIDAY_CONVENTION,USGS
6,COMPOUNDING_METHOD,COMPOUND


In [16]:
factory_fra = qfCreateProductFromDataConvention(
    VALUE_DATE, "USD-SOFR-FRA", f"{EFFECTIVE_DATE}x{FRA_END_DATE}", FIXED_RATE,
    notional=NOTIONAL, pay_or_rec="receive",
)
factory_swap = qfCreateProductFromDataConvention(
    VALUE_DATE, "USD-SOFR-OIS", f"{EFFECTIVE_DATE}x{SWAP_END_DATE}", FIXED_RATE,
    notional=NOTIONAL, pay_or_rec="receive",
)
factory_basis = qfCreateProductFromDataConvention(
    VALUE_DATE, BASIS_CONVENTION, f"{EFFECTIVE_DATE}x{SWAP_END_DATE}", BASIS_SPREAD,
    notional=NOTIONAL, pay_or_rec="receive",
)

for direct, created in [(fra, factory_fra), (swap, factory_swap), (basis_swap, factory_basis)]:
    assert direct.serialize() == created.serialize()
    print(created.product_type, qfCreateValueReport(model, created, parameters, "PV"))

PRODUCT_FRA [['USD', 87.09029420688239]]
PRODUCT_RFR_SWAP [['USD', -2629.6795747881915]]
PRODUCT_OVERNIGHT_INDEX_BASIS_SWAP [['USD', 8446.472471166962]]


## 9. Product values

Use the public valuation API to display each product's PV.


In [17]:
products = {"FRA": fra, "RFR swap": swap, "Overnight basis swap": basis_swap}
pv_rows = [[name, qfCreateValueReport(model, product, parameters, "PV")[0][1]]
           for name, product in products.items()]
display(pd.DataFrame(pv_rows, columns=["Product", "PV"]))



,Product,PV
0,FRA,87.090294
1,RFR swap,"-2,629.679575"
2,Overnight basis swap,"8,446.472471"
